# 01. Spanish Market Structure and BME Competition Restrictions
**Project:** Quantitative Trading Framework — BME Stock Market Contest 2026  
**Objective:** Analyze the market microstructure of the Spanish continuous market (IBEX 35), identify structural liquidity constraints, and mathematically model tournament frictions (minimum €7 brokerage fee and asymmetric BME scoring function).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Analysis environment successfully initialized.")


## 1. Market Overview: Sector Concentration and Structural Oligopoly
The Spanish stock market displays a highly concentrated structure. Unlike broadly diversified indices such as the S&P 500, the IBEX 35 is dominated by a tight cluster of large financial institutions and infrastructure / utility conglomerates.


In [ ]:
# Load market composition metadata
df_market = pd.read_csv('../data/raw/market_composition.csv')
print(f"Total registered assets in universe: {len(df_market)}")
print(f"Constituent assets belonging to IBEX 35: {df_market['Is_IBEX35'].sum()}")
df_market.head(5)


In [ ]:
# 1.1 Top 10 Spanish Companies by Market Capitalization
top_10 = df_market.sort_values(by='MarketCap_B', ascending=False).head(10)

plt.figure(figsize=(12, 5))
bars = plt.barh(top_10['Name'], top_10['MarketCap_B'], color='#1f77b4', edgecolor='black')
plt.xlabel('Market Capitalization (Billions of €)')
plt.title('Top 10 Spanish Companies by Market Capitalization')
plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.8, bar.get_y() + bar.get_height()/2, f'€{width:.1f}B', 
             va='center', ha='left', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 1.2 Sector Distribution of the Spanish Market
sector_mc = df_market.groupby('Sector')['MarketCap_B'].sum().sort_values(ascending=False)
colors = sns.color_palette('Blues_r', len(sector_mc))

plt.figure(figsize=(10, 6))
plt.pie(sector_mc, labels=sector_mc.index, autopct='%1.1f%%', startangle=140, 
        colors=colors, explode=[0.05 if i < 2 else 0 for i in range(len(sector_mc))],
        wedgeprops={'edgecolor': 'black', 'linewidth': 1})
plt.title('Sector Concentration by Market Capitalization in Spain')
plt.tight_layout()
plt.show()

print("Concentration Summary:")
for sector, mc in sector_mc.items():
    print(f" - {sector}: €{mc:.2f}B ({mc / sector_mc.sum() * 100:.1f}%)")


> [!IMPORTANT]
> **Key Structural Finding 1:** Over **60% of total Spanish market capitalization** is concentrated in just two sectors: **Financial Services (Banks)** and **Utilities / Energy**. Designing a quantitative strategy in Spain requires mastering the dynamics of these two industries.


## 2. Liquidity Analysis and Operational Friction Filter
In the BME contest environment, trading small-cap or illiquid assets poses two critical operational risks:
1. **The 2-Minute Execution Rule:** The BME simulator automatically rejects orders that fail to match within 2 minutes due to insufficient order book liquidity.
2. **Slippage and Market Impact:** Executing several thousand euros in thin order books introduces severe market impact.

We evaluate the historical **10th percentile** of daily traded euro volume (`Volume * Close`) over 15 years to benchmark worst-case market liquidity.


In [ ]:
# Load 15-year historical daily closes and volumes
prices = pd.read_csv('../data/raw/stocks_close_15y.csv', index_col=0, parse_dates=True)
volumes = pd.read_csv('../data/raw/stocks_volume_15y.csv', index_col=0, parse_dates=True)

# Daily traded volume in Euros
euro_volume = volumes * prices

# 10th percentile of historical daily euro volume per asset
p10_euro_vol = euro_volume.quantile(0.10)

# Safety threshold: a €10,000 order must not exceed 1% of the 10th percentile daily volume
target_order_size = 10000
slippage_threshold_ratio = 0.01
max_safe_order = p10_euro_vol * slippage_threshold_ratio

liquidity_df = pd.DataFrame({
    'P10_Daily_Euro_Vol': p10_euro_vol,
    'Max_Safe_Order (1%)': max_safe_order
}).dropna()

unsafe_assets = liquidity_df[liquidity_df['Max_Safe_Order (1%)'] < target_order_size].sort_values('P10_Daily_Euro_Vol')
safe_assets = liquidity_df.drop(unsafe_assets.index).sort_values('P10_Daily_Euro_Vol', ascending=False)

print(f"Total evaluated assets: {len(liquidity_df)}")
print(f"Assets passing the institutional liquidity test: {len(safe_assets)}")
print(f"Assets rejected due to illiquidity risk: {len(unsafe_assets)}")


In [ ]:
# Liquidity distribution chart (10th Percentile)
plt.figure(figsize=(14, 6))
plt.bar(safe_assets.index, safe_assets['P10_Daily_Euro_Vol'] / 1e6, color='#2ca02c', label='Eligible for Institutional Execution')
if len(unsafe_assets) > 0:
    plt.bar(unsafe_assets.index, unsafe_assets['P10_Daily_Euro_Vol'] / 1e6, color='#d62728', label='Excluded (Execution / Illiquidity Risk)')

# Cutoff line to absorb a €10,000 order at 1% market participation = €1,000,000 minimum volume
req_vol_millions = (target_order_size / slippage_threshold_ratio) / 1e6
plt.axhline(req_vol_millions, color='black', linestyle='--', linewidth=2, label=f'Minimum Safe Volume (€{req_vol_millions:.0f}M)')

plt.yscale('log')
plt.title('10th Percentile Daily Traded Volume (€ Millions, Logarithmic Scale)')
plt.ylabel('Daily Euro Volume (€ Millions)')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Mathematical Modeling of Brokerage Fees
Official BME competition rules specify a brokerage fee of:
$$\text{Commission} = \max(7.00\text{ €}, 0.15\% \times \text{Notional Value})$$

The fixed **€7 minimum fee** imposes heavy percentage drag on smaller orders:
$$\text{Effective Fee \%} = \frac{\max(7.00, 0.0015 \times V)}{V} \times 100$$


In [ ]:
# Percentage fee drag curve by order notional
order_sizes = np.linspace(200, 15000, 300)
commission_euros = np.maximum(7.0, order_sizes * 0.0015)
commission_pct = (commission_euros / order_sizes) * 100

plt.figure(figsize=(12, 5))
plt.plot(order_sizes, commission_pct, color='#d62728', linewidth=2.5, label='Effective Fee Drag (%)')
plt.axvline(4667, color='black', linestyle=':', label='Optimal Dilution Threshold (€4,667 -> 0.15%)')
plt.axvline(900, color='#ff7f0e', linestyle='--', linewidth=2, label='Micro-Alpha Order Size (€900 -> 0.78% per leg)')

plt.title('Percentage Drag of the Minimum €7 Brokerage Fee by Order Size')
plt.xlabel('Order Size (€)')
plt.ylabel('Effective Cost per Transaction (%)')
plt.ylim(0, 3.5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Fee on a €500 order:   €{max(7.0, 500*0.0015):.2f} ({max(7.0, 500*0.0015)/500*100:.2f}%)")
print(f"Fee on a €900 order:   €{max(7.0, 900*0.0015):.2f} ({max(7.0, 900*0.0015)/900*100:.2f}%)")
print(f"Fee on a €5,000 order: €{max(7.0, 5000*0.0015):.2f} ({max(7.0, 5000*0.0015)/5000*100:.2f}%)")


> [!WARNING]
> **Crucial Quantitative Implication:**
> Active market participants executing 30 to 50 monthly trades in smaller sizing (€1,000 to €2,000) concede **0.35% to 0.70% per leg** (buy + sell = up to **1.40% net round-trip friction**). 
> This structural fee drag rapidly compounds to eliminate any gross intraday edge.


## 4. BME Scoring Function and the Mathematical Floor Effect
The official tournament formula evaluates qualified portfolios up to a theoretical maximum of **9.0 points**:
$$\text{Total Score}_i = \text{Score}_{\text{Return}, i} (\le 6.0) + \text{Score}_{\text{Volatility}, i} (\le 3.0)$$

Where:
$$\text{Score}_{\text{Return}, i} = 6.0 \times \frac{V_{\text{final}, i}}{V_{\max}}$$
$$\text{Score}_{\text{Volatility}, i} = 3.0 \times \left(1 - \frac{\sigma_i - \sigma_{\min}}{\sigma_{\max} - \sigma_{\min}}\right)$$


In [ ]:
# Scoring simulation across typical competitor archetypes
v_initial = 50000.0

profiles = [
    {'Archetype': 'Aggressive Leader (Day Trader)', 'Return_pct': 0.08, 'Annual_Vol': 0.25},
    {'Archetype': 'Trend / Momentum Follower', 'Return_pct': 0.04, 'Annual_Vol': 0.18},
    {'Archetype': 'IBEX 35 Buy & Hold', 'Return_pct': 0.02, 'Annual_Vol': 0.14},
    {'Archetype': 'Average Market Competitor', 'Return_pct': -0.005, 'Annual_Vol': 0.12},
    {'Archetype': 'Surgical Micro-Alpha (Our Model)', 'Return_pct': -0.0024, 'Annual_Vol': 0.0020}
]

df_profiles = pd.DataFrame(profiles)
df_profiles['V_final'] = v_initial * (1 + df_profiles['Return_pct'])

v_max = df_profiles['V_final'].max()
vol_min = df_profiles['Annual_Vol'].min()
vol_max = df_profiles['Annual_Vol'].max()

df_profiles['Score_Return'] = 6.0 * (df_profiles['V_final'] / v_max)
df_profiles['Score_Vol'] = 3.0 * (1 - (df_profiles['Annual_Vol'] - vol_min) / (vol_max - vol_min))
df_profiles['Score_Total'] = df_profiles['Score_Return'] + df_profiles['Score_Vol']

df_profiles = df_profiles.sort_values(by='Score_Total', ascending=False).reset_index(drop=True)
df_profiles[['Archetype', 'Return_pct', 'Annual_Vol', 'Score_Return', 'Score_Vol', 'Score_Total']]


In [ ]:
# Official score decomposition chart
plt.figure(figsize=(12, 6))
bar_width = 0.55
indices = np.arange(len(df_profiles))

p1 = plt.bar(indices, df_profiles['Score_Return'], bar_width, label='Return Score (Max 6.0)', color='#1f77b4')
p2 = plt.bar(indices, df_profiles['Score_Vol'], bar_width, bottom=df_profiles['Score_Return'], label='Volatility Score (Max 3.0)', color='#2ca02c')

plt.xticks(indices, df_profiles['Archetype'], rotation=15, ha='right')
plt.ylabel('Official BME Score (Out of 9.0 Max)')
plt.title('BME Scoring Simulation: The Asymmetric Power of Minimum Volatility')
plt.ylim(0, 10)
plt.axhline(9.0, color='gray', linestyle=':')

for i, total in enumerate(df_profiles['Score_Total']):
    plt.text(i, total + 0.15, f"{total:.2f} pts", ha='center', fontweight='bold', fontsize=11)

plt.legend()
plt.tight_layout()
plt.show()


## 5. Conclusions and Quantitative Foundation

1. **The Spanish Market is an Oligopoly:** Heavy concentration in Banking and Utilities requires filtering out illiquid small-cap noise.
2. **Broker Fees Penalize High Turnover:** The €7 minimum fee demands a surgical, minimal-transaction structure (exactly 3 buys and 3 sells per weekly quota).
3. **The Return Floor Effect:** With €50,000 starting equity, the score deficit conceding to the tournament's top return leader is merely **~0.46 points**, whereas the low-variance advantage is **+3.00 full points**.
4. **Next Step (Notebook 2):** Uncover the temporal anomaly that allows pinning portfolio risk to $\sigma \approx 0.20\%$ while selecting the optimal minimum-variance vehicle (the 5 regulated utilities).
